# Video → trained humanoid policy

One clip in, one trained G1 policy out:
**SAM 3.1** tracks the athlete → MediaPipe 3D pose → mink retargets onto the G1 → **mjlab** trains a PPO motion-tracking policy in MuJoCo Warp → rollout video + metrics.

Runtime: **A100** (or L4 for a slower run). Put your clip in Drive at `a2h/clips/` (2–4 s, one athlete, full body, side view).
Secrets: plain text files on Drive in `a2h/env/`, e.g. `a2h/env/HF_TOKEN` containing `HF_TOKEN = "hf_..."` (token approved for `facebook/sam3.1`). Works from the Colab UI or an IDE attached to a Colab runtime.

## 0. Install (once per runtime, ~5 min)

In [ ]:
REPO_URL = "https://github.com/PandaGrdn/athlete2humanoid.git"

import os, subprocess
# Drive keeps outputs across disconnects. Headless runs (Colab CLI) set A2H_NO_DRIVE=1, since
# mounting needs a browser click, and keep data on the VM instead.
if not os.environ.get("A2H_NO_DRIVE"):
    from google.colab import drive
    drive.mount("/content/drive")
DRIVE_ROOT = "/content/a2h" if os.environ.get("A2H_NO_DRIVE") else "/content/drive/MyDrive/a2h"
REPO = "/content/athlete2humanoid"
for d in ("outputs", "clips"):
    os.makedirs(f"{DRIVE_ROOT}/{d}", exist_ok=True)
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "-q"], check=True)

# Main env: Colab's own Python and numpy 2 (pose, retargeting, rendering).
!pip install -q -r {REPO}/requirements.txt
!bash {REPO}/scripts/fetch_assets.sh

# Headless MuJoCo rendering: Colab only registers Mesa's EGL driver, which can't reach the GPU
# (EGLError, then a dead kernel). Register NVIDIA's, as MuJoCo's own Colab tutorial does.
import json
_egl = "/usr/share/glvnd/egl_vendor.d/10_nvidia.json"
if not os.path.exists(_egl):
    json.dump({"file_format_version": "1.0.0", "ICD": {"library_path": "libEGL_nvidia.so.0"}}, open(_egl, "w"))

# SAM 3.1 and mjlab each get their own virtualenv: SAM pins numpy<2 (no Python 3.13 build),
# mjlab needs its own torch. Neither touches Colab's packages, so no runtime restart.
!pip install -q uv
if not os.path.exists("/content/sam3"):
    !git clone -q https://github.com/facebookresearch/sam3.git /content/sam3
if not os.path.exists("/content/sam_env"):
    !uv venv -q -p 3.12 /content/sam_env
!uv pip install -q -p /content/sam_env/bin/python --torch-backend auto torch torchvision "setuptools<81" -e "/content/sam3[notebooks]"  # sam3 imports pkg_resources
if not os.path.exists("/content/mjlab_env"):
    !uv venv -q -p 3.12 /content/mjlab_env
!uv pip install -q -p /content/mjlab_env/bin/python mjlab==1.6.0 tensorboard
!/content/sam_env/bin/python -c "import torch, sam3; print('SAM env OK, CUDA:', torch.cuda.is_available())"
!/content/mjlab_env/bin/python -c "import torch, mjlab; print('mjlab env OK, CUDA:', torch.cuda.is_available())"

## 1. Session setup (after every restart)

In [ ]:
import os, sys, glob
os.environ["MUJOCO_GL"] = "egl"
DRIVE_ROOT = "/content/a2h" if os.environ.get("A2H_NO_DRIVE") else "/content/drive/MyDrive/a2h"
os.environ.setdefault("A2H_SECRETS_DIR", f"{DRIVE_ROOT}/env")
REPO = "/content/athlete2humanoid"
MJLAB_PY = "/content/mjlab_env/bin/python"
os.environ["A2H_SAM_PYTHON"] = "/content/sam_env/bin/python"
os.environ["A2H_OUTPUT_DIR"] = f"{DRIVE_ROOT}/outputs"
os.environ["A2H_CLIPS_DIR"] = f"{DRIVE_ROOT}/clips"
os.environ["A2H_ASSETS_DIR"] = f"{REPO}/assets"
os.chdir(REPO); sys.path.insert(0, REPO)

import mediapy, numpy as np, torch, config
# Secrets from Drive (a2h/env/HF_TOKEN), so this also works from an IDE attached to the runtime.
config.get_secret("HF_TOKEN")  # exported to os.environ for SAM's subprocess; never printed
print("GPU:", torch.cuda.get_device_name(0))
print("clips on Drive:", sorted(os.path.basename(p) for p in glob.glob(f"{config.CLIPS_DIR}/*")))

## 2. Video → G1 reference motion
Trims the clip, tracks the athlete with SAM 3.1, lifts the pose to 3D, retargets it onto the G1.
Check the side-by-side before training: if the robot doesn't look like the athlete, the policy won't either.

In [ ]:
from pathlib import Path
# Edit these, or set A2H_RAW / A2H_START / A2H_DURATION / A2H_NAME / A2H_PROMPT for headless runs.
RAW = Path(os.environ.get("A2H_RAW", config.CLIPS_DIR / "raw.mp4"))   # your clip (Drive: a2h/clips/)
START = float(os.environ.get("A2H_START", 0.0))                         # seconds to skip
DURATION = float(os.environ.get("A2H_DURATION", 4.0))                   # seconds to keep
NAME = os.environ.get("A2H_NAME", "squat")                              # output folder name
PROMPT = os.environ.get("A2H_PROMPT", "athlete")                        # SAM 3.1 text prompt
USE_SAM = True                         # False = whole frame (only for single-person clips)

if not RAW.exists():
    raise FileNotFoundError(f"No clip at {RAW}. Upload one to Drive (My Drive/a2h/clips/) or set RAW "
                            f"to one of: {sorted(p.name for p in config.CLIPS_DIR.iterdir())}")
CLIP = config.CLIPS_DIR / f"{NAME}.mp4"
!ffmpeg -y -loglevel error -ss {START} -t {DURATION} -i "{RAW}" -vf "fps=30,scale=-2:720" -an "{CLIP}"

import pipeline
result = pipeline.video_to_motion(CLIP, use_sam=USE_SAM, prompt=PROMPT)
OUT = result["out_dir"]
mediapy.show_video(mediapy.read_video(str(OUT / "side_by_side.mp4")), fps=30)

## 3. CSV → mjlab motion (forward kinematics at 50 fps)

In [ ]:
NPZ = OUT / "motion.npz"
!cd /tmp && MUJOCO_GL=egl WANDB_MODE=disabled {MJLAB_PY} -m mjlab.scripts.csv_to_npz --input-file "{OUT}/motion.csv" --output-name {NAME} --input-fps {result['fps']} --output-fps 50
!cp /tmp/motion.npz "{NPZ}"
print({k: v.shape for k, v in np.load(NPZ).items()})

## 4. Train the tracking policy (PPO, 4096 parallel robots)
Watch the iteration time in the first minute and set `ITERS` so it fits your budget.
mjlab's default is 30,000; a few thousand usually gives a first policy that can follow a short clip.
Checkpoints go to Drive, so a disconnect doesn't lose them.

In [ ]:
ITERS = int(os.environ.get("A2H_ITERS", 3000))
LOG_ROOT = f"{DRIVE_ROOT}/logs/{NAME}"
!cd /tmp && MUJOCO_GL=egl WANDB_MODE=disabled /content/mjlab_env/bin/train Mjlab-Tracking-Flat-Unitree-G1 --env.commands.motion.motion-file "{NPZ}" --env.scene.num-envs 4096 --agent.max-iterations {ITERS} --agent.save-interval 500 --agent.logger tensorboard --log-root "{LOG_ROOT}"

## 5. Roll out the trained policy

In [ ]:
ckpts = sorted(glob.glob(f"{LOG_ROOT}/g1_tracking/*/model_*.pt"), key=lambda p: (os.path.getmtime(p), p))
CKPT = ckpts[-1]
print("checkpoint:", CKPT)
!MUJOCO_GL=egl {MJLAB_PY} {REPO}/mjlab_rollout.py --checkpoint "{CKPT}" --motion "{NPZ}" --out "{OUT}"
mediapy.show_video(mediapy.read_video(str(OUT / "policy.mp4")))

## Done when
- `side_by_side.mp4` shows the G1 copying the athlete (kinematic)
- `policy.mp4` shows the G1 doing it **with physics**, and `policy_metrics.json` has `falls: 0`

Everything lands in Drive at `a2h/outputs/<NAME>/` and `a2h/logs/<NAME>/`.

## Optional: Muse Spark API check
Needs `a2h/env/MUSE_API_KEY` on Drive (same format as HF_TOKEN), and a card on the Meta billing page per the guide.

In [ ]:
import motion_brief  # reads a2h/env/MUSE_API_KEY (or META_API_KEY)
if config.get_secret("MUSE_API_KEY", required=False) or config.get_secret("META_API_KEY", required=False):
    print(motion_brief.ask("In one sentence: what are the phases of a vertical squat jump?"))
else:
    print("No MUSE_API_KEY yet; skipping.")